In [1]:
import pandas as pd

DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'

In [2]:
df = pd.read_csv(DATA_PATH)
df.head() # 1 -> Fake

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [3]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

In [4]:
def get_tag(tag:str):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'

def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for token, tag in tagged:
        result = lemmatizer.lemmatize(token, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence : str):
    eng_stopword = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopword
        and
        token.isalpha()
    ]
    
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [5]:
# sentence: extracted feature
# existed words: unique words in corpus
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
        
    return feature

In [6]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [7]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))
    
    # [({feature}, category), ()]  
    features = [
        (feature_extraction(sentence, existed_words), category) 
        for sentence, category in zip(x, y)
    ]
    
    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )
    
    classifier = NaiveBayesClassifier.train(train_data)
    accuracy_percentage = accuracy(classifier, test_data) * 100
    print(f"Accuracy: {accuracy_percentage}%")
    
    return classifier, existed_words

In [8]:
train_classifier()

Accuracy: 75.0%


(<nltk.classify.naivebayes.NaiveBayesClassifier at 0x262251c30e0>,
 {'multithreadingknowledge',
  'emea',
  'reign',
  'employer',
  'post',
  'shipyard',
  'αναπτυσσόμενη',
  'techtool',
  'salaryrelaxed',
  'wl',
  'petabyte',
  'palsy',
  'resultsdevise',
  'sf',
  'sense',
  'vacation',
  'requiredexcellent',
  'structure',
  'procure',
  'workmust',
  'editor',
  'airbrake',
  'andbanqueting',
  'companyrecognised',
  'experienceprevious',
  'arranging',
  'boca',
  'aptitudeextensive',
  'planningzip',
  'share',
  'successfuly',
  'chemistry',
  'moreover',
  'hole',
  'accredit',
  'cuddle',
  'siia',
  'eine',
  'financial',
  'tick',
  'economic',
  'eur',
  'commensurate',
  'sr',
  'integer',
  'capable',
  'prosulting',
  'socialbe',
  'timely',
  'cvs',
  'visibility',
  'count',
  'enough',
  'degreewe',
  'analyzes',
  'detect',
  'partner',
  'await',
  'vouchedfor',
  'timecompletely',
  'outcome',
  'preferredcurrent',
  'έναντι',
  'systemsgenerate',
  'bitbucket',


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [10]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    
    return vectorizer, vectors

In [11]:
import os
import pickle

In [23]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as file:
            classifier, existed_words, vectorizer, vectors = pickle.load(file) 
        return classifier, existed_words, vectorizer, vectors
            
    else:
        print("Model Not Found...")
        print("Training Model...")
        
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()
        
        with open(MODEL_PATH, 'wb') as file:
            pickle.dump((classifier, existed_words, vectorizer, vectors), file)    
        return classifier, existed_words, vectorizer, vectors

In [13]:
def print_menu(sentence, category):
    print('J*b System Real / Fake')
    print(f"Text: {sentence}")
    print(f"Category: {category}")
    print("[1] Write Text")
    print("[2] Recommend")
    print("[3] NER")
    print("[4] Exit")
    
    return input("Input Between 1 - 4: ")

In [14]:
def write_sentence():
    # Min 20 Char and Min 3 Words
    sent = ""
    
    while len(sent.strip()) < 20 or len(sent.strip().split(" ")) < 3:
        sent = input("Input Text Min 20 Char and Min 3 Words: ")
    return sent

In [15]:
def classify_text(classifier, sentence: str, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)
    
    if category == 1:
        return "FAKE"
    else:
        return "REAL"

In [16]:
def load_NER(nlp, sentence):
    doc = nlp(sentence)
    ents_dict = {}
    
    for ent in doc.ents:
        if ent.label_ not in ents_dict.keys():
            ents_dict[ent.label_] = []
        ents_dict[ent.label_].append(ent.text)
    
    return ents_dict

In [17]:
def print_NER(ents_dict):
    
    print("NER")
    
    if ents_dict:
        for key, value in ents_dict.items():
            print(f"{key}: {value}")
    else:
        print("No Entities Found")

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
def recommend_top_n(vectorizer : TfidfVectorizer, job_vectors, query, topn = 5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    top_idx = similarity.argsort()[::-1][:topn]
    
    print("Top 5")
    for i, idx in enumerate(top_idx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [20]:
import spacy

In [21]:
def menu():
    sent = ''
    cat = ''
    
    # Classifier
    classifier = None
    existed_words = None
    
    # Vectorizer
    vectorizer = None
    vectors = None
    
    # N**ER
    nlp = spacy.load("en_core_web_sm")
    entities_dict = {}
    
    while True:
        choice = print_menu(sent, cat).strip()
        print()
        
        if choice == '1':
            sent = write_sentence()
            
            if classifier is None or existed_words is None or vectorizer is None or vectorizer is None:
                classifier, existed_words, vectorizer, vectors = load_model()
                
        elif choice == '2':
            recommend_top_n(vectorizer, vectors, sent)
        
        elif choice == '3':
            if len(sent.strip()) < 1:
                print("No Text Found...")
                
            entities_dict = load_NER(nlp, sent)
            print_NER(entities_dict)
            
        elif choice == '4':  
            print("Thank You")
            return
        
        else:
            print("Invalid Choice Please Try Again...")
        
        print()

In [24]:
menu()

J*b System Real / Fake
Text: 
Category: 
[1] Write Text
[2] Recommend
[3] NER
[4] Exit



c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



J*b System Real / Fake
Text: MBG mas bowo gemoy lol
Category: 
[1] Write Text
[2] Recommend
[3] NER
[4] Exit

Top 5
1. Information Systems Support Specialist  
2. Export Documentation Coordinator/Customer Service
3. URGENT Part Timers & Full Timers Required.
4. Clinic Assistant, Kingston
5. Corporate Accountant CPA

J*b System Real / Fake
Text: MBG mas bowo gemoy lol
Category: 
[1] Write Text
[2] Recommend
[3] NER
[4] Exit

NER
PRODUCT: ['MBG mas bowo']

J*b System Real / Fake
Text: MBG mas bowo gemoy lol
Category: 
[1] Write Text
[2] Recommend
[3] NER
[4] Exit

Thank You
